# Telco churn: imbalanced data, class_weight và calibration

Bộ Telco Customer Churn (IBM, 7 043 khách của một nhà mạng). Khoảng 1/4 khách rời đi (`Churn = Yes`).
Đội retention muốn một **xác suất churn dùng được**: để xếp hạng ai cần gọi, và để ước tính số khách sẽ mất
trong quý (cộng các xác suất lại). Việc thứ hai đòi xác suất **calibrated**, không chỉ thứ hạng tốt.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore`) là chỗ đọc dữ liệu thật, train model, vẽ. Test bỏ qua chúng.
- Chấm: `uv run pytest tracks/data-science/modules/11-classic-ml/exercises/intermediate/02-churn-imbalance-calibration -q`

In [ ]:
import numpy as np
import pandas as pd

## 1. Đọc dữ liệu

`load_telco(path) -> (X, y)`: bỏ `customerID`; `y = (Churn == "Yes")` dạng int; `TotalCharges` là chuỗi và
có 11 ô trống (khách mới, `tenure = 0`): đổi sang số, ô trống thành 0. `X` giữ 19 cột còn lại.

In [ ]:
def load_telco(path) -> tuple[pd.DataFrame, pd.Series]:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: đọc dữ liệu thật (tải bằng: uv run python data/download_dataset.py telco-churn)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
X, y = load_telco(ROOT / "data/raw/telco-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(X.shape, round(y.mean(), 4))
print(pd.crosstab(X.Contract, y, normalize="index").round(3))

In [ ]:
# explore: chia train / calibration / test (phân tầng theo y), dựng pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUM = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
CAT = [c for c in X.columns if c not in NUM]
X_tr, X_rest, y_tr, y_rest = train_test_split(X, y, test_size=0.4, stratify=y, random_state=42)
X_cal, X_te, y_cal, y_te = train_test_split(X_rest, y_rest, test_size=0.5, stratify=y_rest, random_state=42)


def logistic(**kw):
    pre = ColumnTransformer([("num", StandardScaler(), NUM),
                             ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), CAT)])
    return Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, **kw))])


def forest(**kw):
    pre = ColumnTransformer([("num", "passthrough", NUM),
                             ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), CAT)])
    return Pipeline([("pre", pre), ("clf", RandomForestClassifier(300, random_state=0, n_jobs=-1, **kw))])


print(len(X_tr), len(X_cal), len(X_te))

## 2. Đo calibration

`reliability_table(y, p, n_bins=10)`: chia [0, 1] thành `n_bins` khoảng đều (ca `p = 1` vào khoảng cuối),
mỗi khoảng **không rỗng** một dòng, theo thứ tự tăng dần, cột `mean_pred` (trung bình p), `frac_pos`
(tỉ lệ y = 1), `n`. Khớp `sklearn.calibration.calibration_curve(strategy="uniform")`.

`expected_calibration_error(y, p, n_bins=10)`: trung bình có trọng số (theo `n`) của
`|mean_pred − frac_pos|` trên các khoảng.

In [ ]:
def reliability_table(y, p, n_bins: int = 10) -> pd.DataFrame:
    # TODO
    raise NotImplementedError


def expected_calibration_error(y, p, n_bins: int = 10) -> float:
    # TODO: dùng reliability_table
    raise NotImplementedError

## 3. class_weight làm gì với xác suất?

`class_weight="balanced"` (hoặc undersample lớp đa số) thường được dùng "để xử lý imbalance". Nó gần như
không đổi thứ hạng (ROC-AUC), nhưng model học trên một thế giới giả có tỉ lệ dương 50%, nên xác suất bị
phình to. Chạy cell dưới và nhìn cột `mean_p` so với tỉ lệ churn thật.

In [ ]:
# explore: cùng model, có và không có class_weight
fitted = {
    "logistic": logistic().fit(X_tr, y_tr),
    "logistic_balanced": logistic(class_weight="balanced").fit(X_tr, y_tr),
    "forest": forest(min_samples_leaf=10).fit(X_tr, y_tr),
    "forest_balanced": forest(min_samples_leaf=10, class_weight="balanced").fit(X_tr, y_tr),
}
probs = {name: m.predict_proba(X_te)[:, 1] for name, m in fitted.items()}
pd.DataFrame({
    name: {"roc_auc": roc_auc_score(y_te, p), "pr_auc": average_precision_score(y_te, p),
           "brier": brier_score_loss(y_te, p), "ece": expected_calibration_error(y_te, p), "mean_p": p.mean()}
    for name, p in probs.items()
}).T.round(3)

In [ ]:
# explore: reliability diagram
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], "k--", lw=1)
for name in ["logistic", "logistic_balanced", "forest"]:
    t = reliability_table(y_te, probs[name])
    ax.plot(t.mean_pred, t.frac_pos, "o-", label=name)
ax.set_xlabel("xác suất dự đoán (trung bình mỗi khoảng)")
ax.set_ylabel("tỉ lệ churn thật")
ax.legend()
plt.show()

## 4. Sửa xác suất

Hai cách:

1. **Prior correction** khi biết model học trên tỉ lệ dương `train_rate` (0.5 với `class_weight="balanced"`
   hay undersample về 50/50) trong khi tỉ lệ thật là `target_rate`. Theo Bayes, odds bị nhân với
   `(train_rate / (1 − train_rate)) / (target_rate / (1 − target_rate))`; chia lại odds cho hệ số đó.
   `prior_correct(p, train_rate, target_rate)`.
2. **Recalibration** trên một tập calibration riêng (không phải tập train): học một hàm đơn điệu từ score sang
   xác suất. `recalibrate(p_cal, y_cal, p_new, method)`:
   - `"sigmoid"` (Platt): logistic regression một biến trên `logit(p)` (clip p vào [1e-6, 1 − 1e-6]).
   - `"isotonic"`: `sklearn.isotonic.IsotonicRegression(out_of_bounds="clip")`.

   Trả xác suất đã sửa cho `p_new`, nằm trong [0, 1], không đảo thứ hạng.

In [ ]:
def prior_correct(p, train_rate: float, target_rate: float) -> np.ndarray:
    # TODO
    raise NotImplementedError


def recalibrate(p_cal, y_cal, p_new, method: str = "isotonic") -> np.ndarray:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: sửa xác suất của các model balanced bằng ba cách, chấm trên test
rate = y_tr.mean()
rows = {}
for name in ["logistic_balanced", "forest_balanced", "forest"]:
    p_cal = fitted[name].predict_proba(X_cal)[:, 1]
    variants = {"raw": probs[name],
                "sigmoid": recalibrate(p_cal, y_cal, probs[name], "sigmoid"),
                "isotonic": recalibrate(p_cal, y_cal, probs[name], "isotonic")}
    if name.endswith("balanced"):
        variants["prior_correct"] = prior_correct(probs[name], 0.5, rate)
    for how, p in variants.items():
        rows[(name, how)] = {"brier": brier_score_loss(y_te, p), "ece": expected_calibration_error(y_te, p),
                             "mean_p": p.mean(), "roc_auc": roc_auc_score(y_te, p)}
print("tỉ lệ churn thật trên test:", round(y_te.mean(), 3))
pd.DataFrame(rows).T.round(3)

## 5. Câu hỏi để tự trả lời

1. Vì sao ROC-AUC gần như không đổi qua các cách sửa, còn Brier và ECE thay đổi rõ? Isotonic có làm ROC-AUC
   đổi chút ít không, vì sao?
2. Với `class_weight="balanced"`, ngưỡng 0.5 trên xác suất thô tương ứng với ngưỡng bao nhiêu trên xác suất
   đã prior-correct? Đây có phải lý do nhiều người "thấy" class_weight cải thiện recall?
3. Tập calibration có 1 409 khách. Isotonic có nhiều bậc tự do hơn sigmoid. Khi nào bạn chọn cách nào?
4. Đội retention muốn dự báo "quý này mất bao nhiêu khách" bằng tổng xác suất. Model nào trong bảng bạn
   tin để làm việc đó, và bạn kiểm chứng thế nào trước khi dùng?